# 1. How does training shape what an encoder sees?

The concepts NAVE extracts reflect what the encoder learned (paper, Section 4.5 and Table 6). Here
the same image is explained by the same architecture with three trainings:
- random initialisation;
- supervised ImageNet classification;
- self-supervised DINO.

**What to look for:** an untrained encoder still produces regions, but they follow colour and
texture, much like a classical superpixel method. Trained encoders group parts that belong together
semantically (the face, the uniform, the background) even when they look different.

In [ ]:
# %pip install nave-xai
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cbook
from matplotlib.image import imread
from nave_xai import NAVEImage

image = imread(cbook.get_sample_data("grace_hopper.jpg", asfileobj=False))   # ships with matplotlib

def show(ax, explainer, exp, title=""):
    """An explanation map in the explainer's concept colours."""
    ax.imshow(explainer.palette[exp.explanation]); ax.set_title(title, fontsize=10); ax.axis("off")

In [ ]:
settings = [("resnet50", "N34", "random"), ("resnet50", "N34", "imagenet"), ("resnet50", "N34", "dino"),
            ("vits16", "N:12", "random"), ("vits16", "N:12", "imagenet"), ("vits16", "N:12", "dino")]
fig, axes = plt.subplots(2, 3, figsize=(12, 9))
results = {}
for ax, (model, layers, training) in zip(axes.ravel(), settings):
    ex = NAVEImage(model=model, training=training, layers=layers, k=8)
    results[(model, training)] = exp = ex.explain(image)
    show(ax, ex, exp, f"{model} {layers}, {training}")
fig.tight_layout()

### How fragmented are the concepts?

A concept that the encoder understands as one thing tends to form few, large regions.
*Fragmentation* is the number of connected regions per concept: lower means more coherent concepts.

In [ ]:
print(f"{'model':10s} {'training':10s} {'regions/concept':>16s}")
for (model, training), exp in results.items():
    used = len(np.unique(exp.explanation))
    print(f"{model:10s} {training:10s} {(exp.segmentation().max() + 1) / used:16.1f}")

### On your own encoders

The same comparison works for any training you want to audit, e.g. two checkpoints of your model:

```python
for name, model in {"before": model_a, "after": model_b}.items():
    exp = NAVEImage(model, layers=["layer3", "layer4"], k=8).explain(image)
```

## Citation

If you use nave-xai, please cite the paper:

```bibtex
@inproceedings{boubekki2026nave,
  author    = {Boubekki, Ahc{\`e}ne and Fadel, Samuel G. and Mair, Sebastian},
  editor    = {Ceci, Michelangelo and Kocev, Dragi and Monreale, Anna and Vens, Celine and Flach, Peter
               and Manco, Giuseppe and Spiliopoulou, Myra and Masciari, Elio},
  title     = {Multi-depth Concept Extraction for Post-Hoc Vision Encoder Explanation},
  booktitle = {Machine Learning and Knowledge Discovery in Databases. Research Track},
  publisher = {Springer Nature Switzerland},
  address   = {Cham},
  pages     = {449--467},
  year      = {2026},
  doi       = {10.1007/978-3-032-37664-0_26},
  isbn      = {978-3-032-37664-0}
}
```

We thank Ulf Brefeld (Leuphana University Lüneburg) for lending us the computing resources without which this work would not have been possible.